In [ ]:
!pip install transformers scikit-learn nltk

In [ ]:
import pandas as pd
import numpy as np
import torch
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

from transformers import DistilBertTokenizer, DistilBertModel

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
from google.colab import files
uploaded = files.upload()

# Assuming the uploaded file is news_dataset.csv
# You might need to adjust the filename if it's different
import pandas as pd
df = pd.read_csv(list(uploaded.keys())[0])

Saving news_dataset.csv to news_dataset (1).csv


In [ ]:
df['content'] = df['title'] + " " + df['text']

X = df['content']
y = df['label']

df.head()

,Unnamed: 0,title,text,label,content
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1,LAW ENFORCEMENT ON HIGH ALERT Following Threat...
1,1,NaN,Did they post their votes for Hillary already?,1,NaN
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0,"Bobby Jindal, raised Hindu, uses story of Chri..."
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1,SATAN 2: Russia unvelis an image of its terrif...


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
# MODEL 1: TF-IDF + Logistic Regression

tfidf = TfidfVectorizer(stop_words='english', max_df=0.7)

X_train_filled = X_train.fillna('')
X_test_filled = X_test.fillna('')

X_train_tfidf = tfidf.fit_transform(X_train_filled)
X_test_tfidf = tfidf.transform(X_test_filled)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)

pred_tfidf = lr.predict(X_test_tfidf)

print("TF-IDF Accuracy:", accuracy_score(y_test, pred_tfidf))
print("TF-IDF F1 Score:", f1_score(y_test, pred_tfidf))

TF-IDF Accuracy: 0.9473209953559298
TF-IDF F1 Score: 0.9485861182519281


In [ ]:
#MODEL 2: DistilBERT
from transformers import DistilBertTokenizer, DistilBertModel

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')
bert_model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [ ]:
#BERT Embedding Function
def get_embeddings(texts):
    embeddings = []

    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        with torch.no_grad():
            outputs = bert_model(**inputs)

        emb = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
        embeddings.append(emb)

    return np.array(embeddings)

In [ ]:
#Generate Embeddings
X_train_bert = get_embeddings(X_train_filled.tolist()[:1000])
X_test_bert = get_embeddings(X_test_filled.tolist()[:300])

y_train_small = y_train.iloc[:1000]
y_test_small = y_test.iloc[:300]

In [ ]:
#Train BERT Classifier
bert_clf = LogisticRegression(max_iter=1000)
bert_clf.fit(X_train_bert, y_train_small)

pred_bert = bert_clf.predict(X_test_bert)

print("BERT Accuracy:", accuracy_score(y_test_small, pred_bert))
print("BERT F1 Score:", f1_score(y_test_small, pred_bert))

BERT Accuracy: 0.9133333333333333
BERT F1 Score: 0.9182389937106918


In [ ]:
#MODEL 3: HYBRID (TF-IDF + BERT)
tfidf_train_small = X_train_tfidf[:1000].toarray()
tfidf_test_small = X_test_tfidf[:300].toarray()


hybrid_train = np.hstack((tfidf_train_small, X_train_bert))
hybrid_test = np.hstack((tfidf_test_small, X_test_bert))


hybrid_model = LogisticRegression(max_iter=1000)
hybrid_model.fit(hybrid_train, y_train_small)

pred_hybrid = hybrid_model.predict(hybrid_test)

print("Hybrid Accuracy:", accuracy_score(y_test_small, pred_hybrid))
print("Hybrid F1 Score:", f1_score(y_test_small, pred_hybrid))

Hybrid Accuracy: 0.92
Hybrid F1 Score: 0.925


In [ ]:
print("\n📊 FINAL RESULTS")
print("TF-IDF Accuracy:", accuracy_score(y_test, pred_tfidf))
print("BERT Accuracy:", accuracy_score(y_test_small, pred_bert))
print("Hybrid Accuracy:", accuracy_score(y_test_small, pred_hybrid))


📊 FINAL RESULTS
TF-IDF Accuracy: 0.9473209953559298
BERT Accuracy: 0.9133333333333333
Hybrid Accuracy: 0.92


In [ ]:

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
import pickle

pickle.dump(lr, open('model.pkl', 'wb'))
pickle.dump(tfidf, open('tfidf.pkl', 'wb'))

In [ ]:
from google.colab import files

files.download('model.pkl')
files.download('tfidf.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>